In [1]:
# ============================================================
# 1) INSTALLS
# ============================================================
!pip install -q ultralytics opencv-python numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.1 MB/s eta 0:00:00


In [2]:
# ============================================================
# 2) MOUNT DRIVE
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [16]:
# ============================================================
# 3) IMPORTS
# ============================================================
import cv2
import torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO

In [28]:
# ============================================================
# 4) DRIVE PATHS - EDIT ONLY THIS CELL
# ============================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/CV_Models_Final")

MODEL_PATH = PROJECT_ROOT / "models" / "best_unquantized.tflite"
VIDEO_PATH = PROJECT_ROOT / "test_videos" / "car" / "test_video_2.mp4"
OUTPUT_PATH = PROJECT_ROOT / "New_method" / "best_unquantized.mp4"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_PATH:", MODEL_PATH)
print("MODEL exists:", MODEL_PATH.exists())

print("VIDEO_PATH:", VIDEO_PATH)
print("VIDEO exists:", VIDEO_PATH.exists())

print("OUTPUT_PATH:", OUTPUT_PATH)
print("OUTPUT folder exists:", OUTPUT_PATH.parent.exists())
print("OUTPUT folder:", OUTPUT_PATH.parent)

PROJECT_ROOT: /content/drive/MyDrive/CV_Models_Final
MODEL_PATH: /content/drive/MyDrive/CV_Models_Final/models/best_unquantized.tflite
MODEL exists: True
VIDEO_PATH: /content/drive/MyDrive/CV_Models_Final/test_videos/car/test_video_2.mp4
VIDEO exists: True
OUTPUT_PATH: /content/drive/MyDrive/CV_Models_Final/New_method/best_unquantized.mp4
OUTPUT folder exists: True
OUTPUT folder: /content/drive/MyDrive/CV_Models_Final/New_method


In [29]:
# ============================================================
# 5) ORIGINAL LOGIC - DRIVE PATH ADJUSTED ONLY
# ============================================================

BALL_CLASS_ID = 0
CAR_CLASS_ID = 1
FALLEN_PIN_CLASS_ID = 2
STANDING_PIN_CLASS_ID = 3

PIN_CLASSES = [FALLEN_PIN_CLASS_ID, STANDING_PIN_CLASS_ID]

CLASS_COLORS = {
    BALL_CLASS_ID: (0, 255, 255),          # yellow
    CAR_CLASS_ID: (0, 255, 0),             # green
    FALLEN_PIN_CLASS_ID: (0, 0, 255),      # red
    STANDING_PIN_CLASS_ID: (255, 0, 255),  # magenta
}

PATH_COLOR = (0, 0, 255)        # red path
FALL_GLOW_COLOR = (0, 255, 255) # yellow glow


def box_center(box):
    x1, y1, x2, y2 = box
    return int((x1 + x2) / 2), int((y1 + y2) / 2)


def center_distance(box1, box2):
    c1 = box_center(box1)
    c2 = box_center(box2)
    return np.sqrt((c1[0] - c2[0]) ** 2 + (c1[1] - c2[1]) ** 2)


def iou(box1, box2):
    x1, y1, x2, y2 = box1
    a1, b1, a2, b2 = box2

    ix1 = max(x1, a1)
    iy1 = max(y1, b1)
    ix2 = min(x2, a2)
    iy2 = min(y2, b2)

    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)

    inter = iw * ih
    area1 = max(0, x2 - x1) * max(0, y2 - y1)
    area2 = max(0, a2 - a1) * max(0, b2 - b1)

    union = area1 + area2 - inter
    return inter / union if union > 0 else 0


def draw_label(frame, text, x, y, color, font_scale=0.42):
    y = max(y, 20)

    cv2.putText(
        frame,
        text,
        (x, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        font_scale,
        (0, 0, 0),
        2
    )

    cv2.putText(
        frame,
        text,
        (x, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        font_scale,
        color,
        1
    )


def draw_count_panel(frame, standing_count, fallen_count):
    cv2.rectangle(frame, (20, 20), (360, 115), (0, 0, 0), -1)
    cv2.rectangle(frame, (20, 20), (360, 115), (255, 255, 255), 1)

    cv2.putText(
        frame,
        f"Standing pins: {standing_count}",
        (35, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        CLASS_COLORS[STANDING_PIN_CLASS_ID],
        2
    )

    cv2.putText(
        frame,
        f"Fallen pins: {fallen_count}",
        (35, 95),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        CLASS_COLORS[FALLEN_PIN_CLASS_ID],
        2
    )


def draw_fall_timeline(frame, fall_log, frame_width):
    if not fall_log:
        return

    panel_x1 = max(frame_width - 390, 20)
    panel_y1 = 20
    panel_x2 = frame_width - 20
    panel_y2 = min(80 + 25 * len(fall_log), 500)

    cv2.rectangle(frame, (panel_x1, panel_y1), (panel_x2, panel_y2), (0, 0, 0), -1)
    cv2.rectangle(frame, (panel_x1, panel_y1), (panel_x2, panel_y2), (0, 255, 255), 1)

    cv2.putText(
        frame,
        "Fall History",
        (panel_x1 + 15, panel_y1 + 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 255),
        1
    )

    y = panel_y1 + 60

    for entry in sorted(fall_log, key=lambda x: x["order"]):
        text = f"#{entry['order']} Pin {entry['pin_id']} {entry['time']:.2f}s"

        cv2.putText(
            frame,
            text,
            (panel_x1 + 15, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 255),
            1
        )

        y += 24


def detect_car_path_count_pins_with_fall_time(
    project_root=str(PROJECT_ROOT),
    video_name=str(VIDEO_PATH),
    output_name=str(OUTPUT_PATH),
    conf=0.35,
    iou_threshold=0.5,
    imgsz=640,
    max_missing_frames=20,
    match_distance=70,
    match_iou=0.20
):
    PROJECT_ROOT = Path(project_root)

    VIDEO_PATH = Path(video_name)
    OUTPUT_PATH = Path(output_name)
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    BEST_MODEL = Path(MODEL_PATH)
    print("Using model:", BEST_MODEL)

    model = YOLO(str(BEST_MODEL))
    device = 0 if torch.cuda.is_available() else "cpu"

    cap = cv2.VideoCapture(str(VIDEO_PATH))

    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {VIDEO_PATH}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    fps_safe = fps if fps and fps > 0 else 30

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print("Video width:", w)
    print("Video height:", h)
    print("FPS:", fps_safe)
    print("Saving output to:", OUTPUT_PATH)

    out = cv2.VideoWriter(
        str(OUTPUT_PATH),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps_safe,
        (w, h)
    )

    if not out.isOpened():
        raise RuntimeError(f"VideoWriter failed to open output path: {OUTPUT_PATH}")

    car_path = []

    pin_memory = {}
    next_pin_id = 1

    fall_counter = 0
    fall_log = []   # persistent log, never deleted

    frame_index = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frame_index += 1
        used_pin_ids_this_frame = set()

        result = model.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            imgsz=imgsz,
            conf=conf,
            iou=iou_threshold,
            device=device,
            verbose=False
        )[0]

        if result.boxes is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            cls_ids = result.boxes.cls.cpu().numpy().astype(int)
            confs = result.boxes.conf.cpu().numpy()

            track_ids = None
            if result.boxes.id is not None:
                track_ids = result.boxes.id.cpu().numpy().astype(int)

            for idx, box in enumerate(boxes):
                x1, y1, x2, y2 = map(int, box)
                class_id = cls_ids[idx]
                class_name = model.names[class_id]
                score = confs[idx]

                color = CLASS_COLORS.get(class_id, (255, 255, 255))

                if class_id == CAR_CLASS_ID:
                    cx, cy = box_center((x1, y1, x2, y2))
                    car_path.append((cx, cy))

                    label = f"{class_name} {score:.2f}"

                    if track_ids is not None:
                        label += f" ID:{track_ids[idx]}"

                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    draw_label(frame, label, x1, y1 - 8, color)

                elif class_id in PIN_CLASSES:
                    current_box = (x1, y1, x2, y2)

                    best_match_id = None
                    best_match_score = -999

                    for pin_id, pin_data in pin_memory.items():
                        if pin_id in used_pin_ids_this_frame:
                            continue

                        old_box = pin_data["box"]

                        dist = center_distance(current_box, old_box)
                        overlap = iou(current_box, old_box)

                        if dist < match_distance or overlap > match_iou:
                            score_match = overlap - (dist / 1000)

                            if score_match > best_match_score:
                                best_match_score = score_match
                                best_match_id = pin_id

                    if best_match_id is None:
                        best_match_id = next_pin_id
                        next_pin_id += 1

                    used_pin_ids_this_frame.add(best_match_id)

                    old_data = pin_memory.get(best_match_id, {})
                    previous_class = old_data.get("class_id")

                    fall_time = old_data.get("fall_time")
                    fall_order = old_data.get("fall_order")

                    # Detect standing -> fallen
                    if previous_class == STANDING_PIN_CLASS_ID and class_id == FALLEN_PIN_CLASS_ID:
                        if fall_time is None:
                            fall_counter += 1
                            fall_time = frame_index / fps_safe
                            fall_order = fall_counter

                            fall_log.append({
                                "pin_id": best_match_id,
                                "time": fall_time,
                                "order": fall_order
                            })

                    pin_memory[best_match_id] = {
                        "box": current_box,
                        "class_id": class_id,
                        "last_seen": frame_index,
                        "confidence": score,
                    }

                    if fall_time is not None:
                        pin_memory[best_match_id]["fall_time"] = fall_time
                        pin_memory[best_match_id]["fall_order"] = fall_order

                    if class_id == FALLEN_PIN_CLASS_ID:
                        cv2.rectangle(frame, (x1, y1), (x2, y2), FALL_GLOW_COLOR, 4)

                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

                    label = f"{class_name} PIN:{best_match_id}"

                    if class_id == FALLEN_PIN_CLASS_ID and fall_time is not None:
                        label += f" {fall_time:.2f}s #{fall_order}"

                    draw_label(frame, label, x1, y1 - 8, color)

                else:
                    label = f"{class_name} {score:.2f}"

                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    draw_label(frame, label, x1, y1 - 8, color)

        # Clean active memory only, NOT fall_log
        pin_memory = {
            pin_id: data
            for pin_id, data in pin_memory.items()
            if frame_index - data["last_seen"] <= max_missing_frames
        }

        standing_count = sum(
            1 for data in pin_memory.values()
            if data["class_id"] == STANDING_PIN_CLASS_ID
        )

        fallen_count = sum(
            1 for data in pin_memory.values()
            if data["class_id"] == FALLEN_PIN_CLASS_ID
        )

        # Thin solid red path
        for i in range(1, len(car_path)):
            cv2.line(
                frame,
                car_path[i - 1],
                car_path[i],
                PATH_COLOR,
                2,
                lineType=cv2.LINE_AA
            )

        draw_count_panel(frame, standing_count, fallen_count)
        draw_fall_timeline(frame, fall_log, w)

        out.write(frame)

    # =====================================================
    # RELEASE AFTER THE LOOP FINISHES
    # =====================================================
    cap.release()
    out.release()

    print("Total processed frames:", frame_index)
    print("Saved video:", OUTPUT_PATH)
    print("Output exists:", OUTPUT_PATH.exists())

    if OUTPUT_PATH.exists():
        print("Output size MB:", OUTPUT_PATH.stat().st_size / (1024 * 1024))
    else:
        print("ERROR: Output file was not created.")

In [ ]:
# ============================================================
# 6) RUN
# ============================================================
detect_car_path_count_pins_with_fall_time()

Using model: /content/drive/MyDrive/CV_Models_Final/models/best_unquantized.tflite
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Video width: 1024
Video height: 576
FPS: 30.0
Saving output to: /content/drive/MyDrive/CV_Models_Final/New_method/best_unquantized.mp4
Loading /content/drive/MyDrive/CV_Models_Final/models/best_unquantized.tflite for TensorFlow Lite inference...


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
import os, time

os.sync()
time.sleep(3)

print("Files in output folder:")
for p in OUTPUT_PATH.parent.glob("*"):
    print(p, p.stat().st_size / (1024 * 1024), "MB")